# 01 — Davis and KIBA data preparation

## Goal

Reproduce the verified Davis/KIBA pair tables and deterministic random, cold-drug, and cold-target split manifests without changing affinity semantics or source identifiers.

Davis is converted from Kd in nM to pKd. The source KIBA score is retained as a KIBA score and must never be labelled pKd.

**Execution policy:** preprocessing, manifest writing, and figure export are disabled by default. Inspect paths and source checksums before changing a `RUN_*` switch. Run the v2 preprocessing step before training: every processed CSV must travel with its checksum/affinity/source `.metadata.json` sidecar.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import RAW_DATA_DIR, PROCESSED_DATA_DIR, SPLITS_DIR

DATASETS = ("davis", "kiba")
SPLIT_TYPES = ("random", "cold_drug", "cold_target")
SEEDS = (42, 123, 2026)

RUN_PREPROCESSING = False
RUN_MANIFESTS = False
RUN_MOLECULAR_GRAPH_FIGURE = False

print(json.dumps({
    "datasets": DATASETS,
    "splits": SPLIT_TYPES,
    "seeds": SEEDS,
    "raw_root": str(RAW_DATA_DIR),
    "processed_root": str(PROCESSED_DATA_DIR),
    "manifest_root": str(SPLITS_DIR / "manifests"),
}, indent=2))


## Steps

`prepare_dataset` preserves the verified loader, transformation, validation, and stable pair-ID contract. The summary should include row/entity counts, target semantics, source hashes, and rejected-record counts. It writes `<dataset>_pairs.csv` and `<dataset>_pairs.metadata.json` as one provenance unit; legacy processed CSVs without the sidecar are intentionally rejected rather than silently trusted.


In [ ]:
if RUN_PREPROCESSING:
    prepare_dataset = require_api("src.preprocessing", "prepare_dataset")
    preparation_summaries = {}
    for dataset_name in DATASETS:
        output_path = PROCESSED_DATA_DIR / f"{dataset_name}_pairs.csv"
        pairs, summary = prepare_dataset(
            dataset_name=dataset_name,
            raw_data_directory=RAW_DATA_DIR / dataset_name,
            output_path=output_path,
        )
        metadata_path = output_path.with_suffix(".metadata.json")
        if not output_path.exists() or not metadata_path.exists():
            raise FileNotFoundError(
                "prepare_dataset must create both the processed CSV and "
                f"its provenance sidecar: {output_path}, {metadata_path}"
            )
        preparation_summaries[dataset_name] = summary
        print({
            "dataset": dataset_name,
            "pairs": len(pairs),
            "processed_csv": str(output_path),
            "metadata_sidecar": str(metadata_path),
        })
else:
    print("Preprocessing skipped: RUN_PREPROCESSING is False.")


In [ ]:
if RUN_MANIFESTS:
    import pandas as pd

    materialize = require_api("src.splits", "materialize_split_manifests")
    for dataset_name in DATASETS:
        pairs_path = PROCESSED_DATA_DIR / f"{dataset_name}_pairs.csv"
        if not pairs_path.exists():
            raise FileNotFoundError(f"Prepare {pairs_path} first.")
        metadata_path = pairs_path.with_suffix(".metadata.json")
        if not metadata_path.exists():
            raise FileNotFoundError(
                f"Missing provenance sidecar {metadata_path}. Rerun the v2 "
                "prepare_dataset step; legacy CSVs fail closed."
            )
        pairs = pd.read_csv(pairs_path)
        for seed in SEEDS:
            metadata = materialize(
                dataframe=pairs,
                dataset_name=dataset_name,
                output_root=SPLITS_DIR / "manifests",
                seed=seed,
            )
            print(dataset_name, seed, metadata)
else:
    print("Split-manifest generation skipped: RUN_MANIFESTS is False.")


In [ ]:
if RUN_MOLECULAR_GRAPH_FIGURE:
    visualise = require_api("src.visualization", "plot_molecule_and_graph")
    example_smiles = "CC(=O)Oc1ccccc1C(=O)O"  # aspirin; figure example only
    figure, _ = visualise(example_smiles)
    output_path = PROJECT_ROOT / "results" / "plots" / "molecule_graph_example.png"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    print(output_path)
else:
    print("Molecular-graph figure skipped: RUN_MOLECULAR_GRAPH_FIGURE is False.")


## Checks

Before training, verify dataset row counts, pair-ID uniqueness, finite affinities, valid SMILES, expected pKd/KIBA labels, split conservation, and all pair/entity-overlap diagnostics. Identifier-disjoint cold splits do not guarantee structure- or sequence-disjointness.

## Next steps

Run `pytest -q`, then configure exactly one experiment in notebook 02 or 05.
